# Distributed Vectors

This notebook covers working with DTL distributed vectors in detail.

## Topics
- Creating vectors with different dtypes
- Understanding data partitioning
- Working with local views
- Fill operations

In [ ]:
import dtl
import numpy as np

## Supported Data Types

DTL vectors support multiple NumPy dtypes:

In [ ]:
with dtl.Context() as ctx:
    # Floating point
    vec_f64 = dtl.DistributedVector(ctx, size=100, dtype=np.float64)
    vec_f32 = dtl.DistributedVector(ctx, size=100, dtype=np.float32)
    
    # Signed integers
    vec_i64 = dtl.DistributedVector(ctx, size=100, dtype=np.int64)
    vec_i32 = dtl.DistributedVector(ctx, size=100, dtype=np.int32)
    
    # Unsigned integers
    vec_u64 = dtl.DistributedVector(ctx, size=100, dtype=np.uint64)
    vec_u32 = dtl.DistributedVector(ctx, size=100, dtype=np.uint32)
    vec_u8 = dtl.DistributedVector(ctx, size=100, dtype=np.uint8)
    
    # Check dtypes
    print(f"float64 vector dtype: {vec_f64.local_view().dtype}")
    print(f"float32 vector dtype: {vec_f32.local_view().dtype}")
    print(f"int64 vector dtype: {vec_i64.local_view().dtype}")
    print(f"uint8 vector dtype: {vec_u8.local_view().dtype}")

## Data Partitioning

DTL uses block partitioning by default. Each rank gets a contiguous chunk of the data.

In [ ]:
with dtl.Context() as ctx:
    # Create vector
    vec = dtl.DistributedVector(ctx, size=1000, dtype=np.float64)
    
    print(f"Global size: {vec.global_size}")
    print(f"Number of ranks: {ctx.size}")
    print(f"\nRank {ctx.rank}:")
    print(f"  Local size: {vec.local_size}")
    print(f"  Local offset: {vec.local_offset}")
    print(f"  Index range: [{vec.local_offset}, {vec.local_offset + vec.local_size})")

## Working with Local Views

The `local_view()` method returns a NumPy array that shares memory with the vector.

In [ ]:
with dtl.Context() as ctx:
    vec = dtl.DistributedVector(ctx, size=100, dtype=np.float64)
    
    # Get local view
    local = vec.local_view()
    
    # Fill using global indices
    global_indices = np.arange(vec.local_offset, vec.local_offset + vec.local_size)
    local[:] = global_indices ** 2  # Each element is its global index squared
    
    print(f"Rank {ctx.rank}: elements {vec.local_offset} to {vec.local_offset + vec.local_size - 1}")
    print(f"First 5 values: {local[:5]}")

## NumPy Operations

Since `local_view()` returns a real NumPy array, all NumPy operations work:

In [ ]:
with dtl.Context() as ctx:
    vec = dtl.DistributedVector(ctx, size=1000, dtype=np.float64)
    local = vec.local_view()
    
    # Initialize with random data
    np.random.seed(42 + ctx.rank)  # Different seed per rank
    local[:] = np.random.randn(len(local))
    
    # NumPy operations
    print(f"Rank {ctx.rank}:")
    print(f"  Min: {local.min():.4f}")
    print(f"  Max: {local.max():.4f}")
    print(f"  Mean: {local.mean():.4f}")
    print(f"  Std: {local.std():.4f}")

## Fill Operations

Use `fill()` to set all elements to a value, or create with a fill value:

In [ ]:
with dtl.Context() as ctx:
    # Create with fill value
    vec1 = dtl.DistributedVector(ctx, size=100, dtype=np.float64, fill=42.0)
    print(f"Created with fill: {vec1.local_view()[:5]}")
    
    # Fill after creation
    vec2 = dtl.DistributedVector(ctx, size=100, dtype=np.float64)
    vec2.fill(99.0)
    print(f"Filled after creation: {vec2.local_view()[:5]}")

## Computing Local Statistics

Example: compute local sum and prepare for distributed reduction

In [ ]:
with dtl.Context() as ctx:
    # Create and initialize
    vec = dtl.DistributedVector(ctx, size=10000, dtype=np.float64)
    local = vec.local_view()
    
    # Each element is 1.0
    local[:] = 1.0
    
    # Compute local sum
    local_sum = np.sum(local)
    print(f"Rank {ctx.rank}: local_sum = {local_sum}")
    
    # For global sum, use mpi4py
    try:
        from mpi4py import MPI
        global_sum = MPI.COMM_WORLD.allreduce(local_sum, op=MPI.SUM)
        print(f"Global sum: {global_sum} (expected: {vec.global_size})")
    except ImportError:
        print("(mpi4py not available for global reduction)")